# 다중 특성 회귀 모델 파이프라인
### WHO 기대수명 예측 | End-to-End ML 과제
**실행 순서: 위에서 아래로 셀을 순서대로 실행하세요.**

In [ ]:
# ── 0. 패키지 설치 (Colab 최초 실행 시) ─────────────
!pip install streamlit pyngrok joblib scikit-learn -q

In [ ]:
# ── 1. 라이브러리 임포트 ─────────────────────────────
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

from sklearn.pipeline        import Pipeline
from sklearn.preprocessing   import StandardScaler, PolynomialFeatures
from sklearn.linear_model    import LinearRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics         import r2_score, mean_squared_error

print('✅ 라이브러리 임포트 완료')

In [ ]:
# ── 2. 데이터 로드 및 결측치 제거 ───────────────────
URL = 'https://github.com/dongupak/DataML/raw/main/csv/life_expectancy.csv'
df  = pd.read_csv(URL).dropna()

print(f'데이터 크기: {df.shape}')
print('컬럼 목록:')
print(df.columns.tolist())

In [ ]:
# ── 3. 특성 선택 (Schooling 제외, 3개 이상 자율 선택) ─
FEATURES = ['Adult Mortality', 'BMI', 'GDP']   # ← 원하면 변경 가능
TARGET   = 'Life expectancy '                   # 컬럼명 끝 공백 주의

X = df[FEATURES]
y = df[TARGET]

print(f'독립변수: {FEATURES}')
print(f'종속변수: {TARGET.strip()}')
X.describe()

In [ ]:
# ── 4. Train/Test 분할 + 50개 샘플링 ────────────────
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 훈련 데이터 50개만 추출 (과대적합 관찰용)
np.random.seed(42)
idx     = np.random.choice(len(X_train_full), size=50, replace=False)
X_train = X_train_full.iloc[idx]
y_train = y_train_full.iloc[idx]

print(f'학습 샘플 : {X_train.shape[0]}개')
print(f'테스트 샘플: {X_test.shape[0]}개')

In [ ]:
# ── 5. 3가지 파이프라인 정의 ─────────────────────────

# Model 1 : 1차 선형 회귀
pipe_linear = Pipeline([
    ('scaler', StandardScaler()),
    ('reg',    LinearRegression())
])

# Model 2 : 3차 다항 회귀 (규제 없음 → 과대적합 유도)
pipe_poly = Pipeline([
    ('scaler', StandardScaler()),
    ('poly',   PolynomialFeatures(degree=3, include_bias=False)),
    ('reg',    LinearRegression())
])

# Model 3 : 3차 다항 + Ridge 규제 (alpha=1.0)
pipe_ridge = Pipeline([
    ('scaler', StandardScaler()),
    ('poly',   PolynomialFeatures(degree=3, include_bias=False)),
    ('reg',    Ridge(alpha=1.0))
])

# 학습
pipe_linear.fit(X_train, y_train)
pipe_poly.fit(X_train,   y_train)
pipe_ridge.fit(X_train,  y_train)

print('✅ 3가지 모델 학습 완료')

In [ ]:
# ── 6. 성능 평가 및 테이블 생성 ─────────────────────
models_dict = {'Linear': pipe_linear, 'Poly': pipe_poly, 'Ridge': pipe_ridge}
rows = []

for name, pipe in models_dict.items():
    y_tr  = pipe.predict(X_train)
    y_te  = pipe.predict(X_test)
    compl = (pipe.named_steps['poly'].n_output_features_
             if 'poly' in pipe.named_steps else len(FEATURES))
    rows.append({
        'Model':      name,
        'Train R²':  round(r2_score(y_train, y_tr), 4),
        'Test R²':   round(r2_score(y_test,  y_te), 4),
        'Train MSE': round(mean_squared_error(y_train, y_tr), 2),
        'Test MSE':  round(mean_squared_error(y_test,  y_te), 2),
        'Complexity': compl,
    })

result_df = pd.DataFrame(rows)
print(result_df.to_string(index=False))

In [ ]:
# ── 7. Test R² 막대그래프 ───────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
colors  = ['#378ADD', '#E24B4A', '#3B6D11']
bars    = ax.bar(result_df['Model'], result_df['Test R²'],
                 color=colors, width=0.45, edgecolor='none')
ax.bar_label(bars, fmt='%.4f', padding=4, fontsize=11)
ax.set_title('Test R² Score — Model Comparison', fontsize=14, pad=10)
ax.set_ylabel('R² Score')
ax.set_ylim(min(result_df['Test R²'].min() - 0.15, -0.2), 1.1)
ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=120)
plt.show()
print('✅ model_comparison.png 저장 완료')

In [ ]:
# ── 8. 모델 및 메타데이터 저장 ──────────────────────
joblib.dump(pipe_linear, 'model_linear.pkl')
joblib.dump(pipe_poly,   'model_poly.pkl')
joblib.dump(pipe_ridge,  'model_ridge.pkl')

feature_info = {
    'features': FEATURES,
    'ranges':   {f: (float(X[f].min()), float(X[f].max())) for f in FEATURES},
    'means':    {f: float(X[f].mean()) for f in FEATURES},
}
joblib.dump(feature_info, 'feature_info.pkl')

result_df.to_csv('performance.csv', index=False)

print('✅ 저장 완료:')
print('   model_linear.pkl, model_poly.pkl, model_ridge.pkl')
print('   feature_info.pkl, performance.csv')

In [ ]:
# ── 9. app.py 파일 생성 ──────────────────────────────
app_code = r'''
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

st.set_page_config(page_title="기대수명 예측 대시보드", layout="wide")

@st.cache_resource
def load_models():
    return {
        "Linear": joblib.load("model_linear.pkl"),
        "Poly":   joblib.load("model_poly.pkl"),
        "Ridge":  joblib.load("model_ridge.pkl"),
    }

@st.cache_resource
def load_feature_info():
    return joblib.load("feature_info.pkl")

models    = load_models()
feat_info = load_feature_info()
FEATURES  = feat_info["features"]
RANGES    = feat_info["ranges"]
MEANS     = feat_info["means"]

st.sidebar.title("입력값 설정")
st.sidebar.markdown("---")
user_input = {}
for feat in FEATURES:
    mn, mx = RANGES[feat]
    user_input[feat] = st.sidebar.slider(
        feat, float(mn), float(mx), float(MEANS[feat]),
        step=round((mx - mn) / 100, 2)
    )

st.sidebar.markdown("---")
model_name = st.sidebar.selectbox("모델 선택", ["Linear", "Poly", "Ridge"])

st.title("기대수명 예측 대시보드")
st.caption("WHO Life Expectancy 데이터 기반 다중 특성 회귀 파이프라인")
st.markdown("---")

X_input = pd.DataFrame([user_input])
pred    = round(float(max(0, min(models[model_name].predict(X_input)[0], 100))), 2)

c1, c2, c3 = st.columns(3)
c1.metric("예측 기대수명", f"{pred} 세", f"{model_name} 모델")
c2.metric("선택 모델", model_name)
c3.metric("입력 특성 수", len(FEATURES))

st.markdown("---")
st.subheader("모델 성능 비교")

try:
    perf_df = pd.read_csv("performance.csv")
    st.dataframe(perf_df, use_container_width=True)
    fig, ax = plt.subplots(figsize=(6, 3.5))
    bars = ax.bar(perf_df["Model"], perf_df["Test R²"],
                  color=["#378ADD", "#E24B4A", "#3B6D11"], width=0.45, edgecolor="none")
    ax.bar_label(bars, fmt="%.4f", padding=4, fontsize=11)
    ax.set_title("Test R² Score Comparison", fontsize=13)
    ax.set_ylabel("R² Score")
    ax.set_ylim(min(perf_df["Test R²"].min() - 0.15, -0.2), 1.1)
    ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    st.pyplot(fig)
    plt.close(fig)
except FileNotFoundError:
    st.warning("performance.csv 없음 - train_models.ipynb를 먼저 실행하세요.")

st.markdown("---")
st.caption("선택 독립변수: " + " · ".join(FEATURES))
'''

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(app_code.strip())
print('✅ app.py 생성 완료')

## 4단계 — Streamlit + ngrok 실행
아래 셀에서 **본인 ngrok 토큰**을 입력한 뒤 실행하세요.  
ngrok 토큰 발급: https://dashboard.ngrok.com → Your Authtoken

In [ ]:
import subprocess, threading, time
from pyngrok import ngrok, conf

# ▼▼▼ 여기에 본인 ngrok 토큰 입력 ▼▼▼
NGROK_TOKEN = 'YOUR_NGROK_TOKEN_HERE'
# ▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲

conf.get_default().auth_token = NGROK_TOKEN

def run_streamlit():
    subprocess.run([
        'streamlit', 'run', 'app.py',
        '--server.port', '8501',
        '--server.headless', 'true'
    ])

t = threading.Thread(target=run_streamlit, daemon=True)
t.start()
time.sleep(6)

public_url = ngrok.connect(8501)
print(f'\n✅ 공개 URL: {public_url}')
print('\n위 URL을 모바일/외부 PC로 접속하여 화면을 캡처하세요!')

## 스크린샷 첨부 (여기에 붙여넣기)

모바일/외부 PC로 접속한 화면 캡처를 아래 셀에 삽입하세요.  
(Colab: 삽입 → 이미지 또는 셀에 마크다운으로 `![](이미지경로)` 형식)

In [ ]:
# 스크린샷 표시 예시
from IPython.display import Image, display
# display(Image('screenshot.png'))  # 캡처 파일명으로 변경